# 06 — Final Labeled Lip Dataset Builder (Manifest-Only)

สร้าง training manifest จากข้อมูล final โดย **ไม่ copy รูป mouth ซ้ำ** เพื่อประหยัดพื้นที่ Drive D

Input:
- `dataset/final_alignment/video001/final_speaker_transcript_auto.csv`
- `dataset/mouth_crops_final/video001/mouth_metadata.csv`

Output:
- `dataset/final_dataset/video001/segment_qa.csv`
- `dataset/final_dataset/video001/manifest_train_ready.csv`
- `dataset/final_dataset/video001/accepted_segments.csv`
- `dataset/final_dataset/video001/rejected_segments.csv`

## Cell 1 — Path

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

ALIGNMENT_CSV = PROJECT_ROOT / "dataset" / "final_alignment" / VIDEO_ID / "final_speaker_transcript_auto.csv"
MOUTH_METADATA_CSV = PROJECT_ROOT / "dataset" / "mouth_crops_final" / VIDEO_ID / "mouth_metadata.csv"
OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "final_dataset" / VIDEO_ID

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("ALIGNMENT_CSV     :", ALIGNMENT_CSV)
print("MOUTH_METADATA_CSV:", MOUTH_METADATA_CSV)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)

if not ALIGNMENT_CSV.exists():
    raise FileNotFoundError(ALIGNMENT_CSV)
if not MOUTH_METADATA_CSV.exists():
    raise FileNotFoundError(MOUTH_METADATA_CSV)

ALIGNMENT_CSV     : D:\Face_detection\dataset\final_alignment\video001\final_speaker_transcript_auto.csv
MOUTH_METADATA_CSV: D:\Face_detection\dataset\mouth_crops_final\video001\mouth_metadata.csv
OUTPUT_ROOT       : D:\Face_detection\dataset\final_dataset\video001


## Cell 2 — Load

In [2]:
import pandas as pd
import numpy as np
import re

align_df = pd.read_csv(ALIGNMENT_CSV)
mouth_df = pd.read_csv(MOUTH_METADATA_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = mouth_df["accepted"].astype(str).str.lower().isin(["true","1","yes"])

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()

for c in ["frame_index","speaker_id"]:
    mouth_df[c] = mouth_df[c].astype(int)

for c in ["segment_id","speaker_id","start_frame","end_frame"]:
    align_df[c] = align_df[c].astype(int)

print("AUTO segments      :", len(align_df))
print("Accepted mouth rows:", len(mouth_df))
print("Speakers           :", sorted(align_df["speaker_id"].unique().tolist()))
display(align_df.head())

AUTO segments      : 771
Accepted mouth rows: 253684
Speakers           : [0, 1]


,video_id,segment_id,start_time,end_time,start_frame,end_frame,text,speaker_id,speaker_score,second_score,relative_margin,coverage,mean_motion,status
0,video001,10,50.76,52.78,1521,1582,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้,0,0.173439,0.0,1.0,1.0,0.023734,AUTO
1,video001,11,52.78,55.40,1581,1661,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน,0,0.172227,0.0,1.0,1.0,0.023782,AUTO
2,video001,12,55.40,56.92,1660,1706,อย่างเช่นการเอาไปอับเงินเดือน,0,0.176816,0.0,1.0,1.0,0.025016,AUTO
3,video001,13,56.92,58.76,1705,1762,หรือการเติบโตในหน้าที่การงาน,0,0.170431,0.0,1.0,1.0,0.021170,AUTO
4,video001,16,67.58,69.54,2025,2085,เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ,1,0.176910,0.0,1.0,1.0,0.028995,AUTO


## Cell 3 — QA Thresholds

In [3]:
MIN_DURATION_SEC = 0.40
MAX_DURATION_SEC = 8.00
MIN_FRAMES = 10
MIN_COVERAGE = 0.85
MIN_CONTINUITY = 0.90
MIN_TEXT_LENGTH = 2
MAX_TEXT_LENGTH = 180

print("Thresholds ready")

Thresholds ready


## Cell 4 — Clean Text

In [4]:
def clean_text(text):
    return re.sub(r"\s+", " ", str(text).strip())

def suspicious_text(text):
    text = clean_text(text)
    thai_chars = len(re.findall(r"[ก-๙]", text))
    alpha_chars = len(re.findall(r"[A-Za-zก-๙]", text))
    if alpha_chars == 0:
        return True
    return (thai_chars / alpha_chars) < 0.50

align_df["text_clean"] = align_df["text"].apply(clean_text)
align_df["text_length"] = align_df["text_clean"].str.len()
align_df["suspicious_text"] = align_df["text_clean"].apply(suspicious_text)

display(align_df[["segment_id","text_clean","text_length","suspicious_text"]].head(30))

,segment_id,text_clean,text_length,suspicious_text
0,10,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้,42,False
1,11,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน,33,False
2,12,อย่างเช่นการเอาไปอับเงินเดือน,29,False
3,13,หรือการเติบโตในหน้าที่การงาน,28,False
4,16,เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ,34,False
5,17,เหมือนเรามีครูส่วนตัวที่ไปทุกที่กับเรา,38,False
6,18,เรียนได้แบบ 24 ชั่วโมงเลย สะดวกมาก,34,False
7,19,ใช่ค่ะ,6,False
8,20,คือพี่ลองเล่นมาแล้ว แล้วพี่ชอบที่เขาจะมี AI,43,False
9,21,เหมือนก็ว่ามาประเมินก่อน,24,False


## Cell 5 — Segment QA

In [5]:
qa_rows = []

for _, seg in align_df.iterrows():
    segment_id = int(seg["segment_id"])
    speaker_id = int(seg["speaker_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    start_time = float(seg["start_time"])
    end_time = float(seg["end_time"])
    duration = end_time - start_time
    text = str(seg["text_clean"])

    expected_frames = max(1, end_frame - start_frame + 1)

    matched = mouth_df[
        (mouth_df["speaker_id"] == speaker_id) &
        (mouth_df["frame_index"] >= start_frame) &
        (mouth_df["frame_index"] <= end_frame)
    ].copy().sort_values("frame_index")

    frame_indices = matched["frame_index"].drop_duplicates().astype(int).tolist()
    observed_frames = len(frame_indices)
    coverage = observed_frames / expected_frames if expected_frames else 0.0

    if observed_frames <= 1:
        continuity = 0.0
    else:
        diffs = np.diff(frame_indices)
        continuity = float(np.mean(diffs <= 2))

    reasons = []
    if duration < MIN_DURATION_SEC: reasons.append("too_short")
    if duration > MAX_DURATION_SEC: reasons.append("too_long")
    if observed_frames < MIN_FRAMES: reasons.append("too_few_frames")
    if coverage < MIN_COVERAGE: reasons.append("low_coverage")
    if continuity < MIN_CONTINUITY: reasons.append("low_continuity")
    if len(text) < MIN_TEXT_LENGTH: reasons.append("text_too_short")
    if len(text) > MAX_TEXT_LENGTH: reasons.append("text_too_long")
    if bool(seg["suspicious_text"]): reasons.append("suspicious_text")

    qa_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "speaker_id": speaker_id,
        "start_time": start_time,
        "end_time": end_time,
        "duration": round(duration, 3),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "expected_frames": expected_frames,
        "observed_frames": observed_frames,
        "coverage": round(coverage, 4),
        "continuity": round(continuity, 4),
        "text": text,
        "status": "ACCEPT" if not reasons else "REJECT",
        "reject_reason": "|".join(reasons),
    })

qa_df = pd.DataFrame(qa_rows)

print(qa_df["status"].value_counts())
display(qa_df.head(30))

status
ACCEPT    732
REJECT     39
Name: count, dtype: int64


,video_id,segment_id,speaker_id,start_time,end_time,duration,start_frame,end_frame,expected_frames,observed_frames,coverage,continuity,text,status,reject_reason
0,video001,10,0,50.76,52.78,2.02,1521,1582,62,62,1.0,1.0,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้,ACCEPT,
1,video001,11,0,52.78,55.40,2.62,1581,1661,81,81,1.0,1.0,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน,ACCEPT,
2,video001,12,0,55.40,56.92,1.52,1660,1706,47,47,1.0,1.0,อย่างเช่นการเอาไปอับเงินเดือน,ACCEPT,
3,video001,13,0,56.92,58.76,1.84,1705,1762,58,58,1.0,1.0,หรือการเติบโตในหน้าที่การงาน,ACCEPT,
4,video001,16,1,67.58,69.54,1.96,2025,2085,61,61,1.0,1.0,เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ,ACCEPT,
5,video001,17,1,69.54,72.02,2.48,2084,2159,76,76,1.0,1.0,เหมือนเรามีครูส่วนตัวที่ไปทุกที่กับเรา,ACCEPT,
6,video001,18,1,72.02,74.50,2.48,2158,2233,76,76,1.0,1.0,เรียนได้แบบ 24 ชั่วโมงเลย สะดวกมาก,ACCEPT,
7,video001,19,1,74.50,75.48,0.98,2232,2263,32,32,1.0,1.0,ใช่ค่ะ,ACCEPT,
8,video001,20,1,75.48,78.52,3.04,2262,2354,93,93,1.0,1.0,คือพี่ลองเล่นมาแล้ว แล้วพี่ชอบที่เขาจะมี AI,ACCEPT,
9,video001,21,1,78.52,79.88,1.36,2353,2395,43,43,1.0,1.0,เหมือนก็ว่ามาประเมินก่อน,ACCEPT,


## Cell 6 — Reject Summary

In [6]:
reject_df = qa_df[qa_df["status"] == "REJECT"].copy()

reason_counts = {}
for value in reject_df["reject_reason"].fillna(""):
    for reason in str(value).split("|"):
        if reason:
            reason_counts[reason] = reason_counts.get(reason, 0) + 1

reason_summary = pd.DataFrame(
    list(reason_counts.items()),
    columns=["reason","count"]
).sort_values("count", ascending=False)

display(reason_summary)
display(reject_df[[
    "segment_id","speaker_id","duration","observed_frames",
    "coverage","continuity","text","reject_reason"
]].head(60))

,reason,count
0,too_long,22
2,low_coverage,8
1,suspicious_text,7
3,too_short,5
5,too_few_frames,2
4,text_too_long,1
6,low_continuity,1


,segment_id,speaker_id,duration,observed_frames,coverage,continuity,text,reject_reason
15,27,1,11.74,353,1.0000,1.0000,ทั้ง Coach การออกเสียง,too_long
20,32,1,2.46,75,1.0000,1.0000,แล้วพี่ชอบ Feature ที่เป็น Speech Analyzer มาก,suspicious_text
29,41,1,2.30,71,1.0000,1.0000,Roleplay กับ AI ใน APP ด้วยนะ,suspicious_text
42,58,0,2.20,68,1.0000,1.0000,และเรื่อง Package Premium หรือ Family,suspicious_text
46,69,0,1.40,28,0.6364,1.0000,เขามาถามว่า,low_coverage
48,73,0,10.72,323,1.0000,1.0000,ในช่วงแรกที่คุมกับแฟน,too_long
115,188,0,8.18,247,1.0000,1.0000,เรามีความชอบของหนัก,too_long
123,198,0,0.40,14,1.0000,1.0000,ใช่,too_short
136,240,1,13.70,412,1.0000,1.0000,ต้องเปลี่ยนชุดว้ายน้ำและลงเล่นน้ำ,too_long
144,248,1,13.92,419,1.0000,1.0000,แต่ถึงขั้นบังคับให้เจ้าของโพส,too_long


## Cell 7 — Build Manifest Without Copying Images

In [7]:
manifest_rows = []

accepted_df = qa_df[qa_df["status"] == "ACCEPT"].copy()

for _, seg in accepted_df.iterrows():
    speaker_id = int(seg["speaker_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    segment_id = int(seg["segment_id"])

    matched = mouth_df[
        (mouth_df["speaker_id"] == speaker_id) &
        (mouth_df["frame_index"] >= start_frame) &
        (mouth_df["frame_index"] <= end_frame)
    ].copy().sort_values("frame_index")

    frame_paths = matched["mouth_file"].astype(str).tolist()
    frame_indices = matched["frame_index"].astype(int).tolist()

    if not frame_paths:
        continue

    clip_id = f"{VIDEO_ID}_seg{segment_id:06d}_spk{speaker_id:03d}"

    manifest_rows.append({
        "clip_id": clip_id,
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "speaker_id": speaker_id,
        "start_time": float(seg["start_time"]),
        "end_time": float(seg["end_time"]),
        "duration": float(seg["duration"]),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "num_frames": len(frame_paths),
        "coverage": float(seg["coverage"]),
        "continuity": float(seg["continuity"]),
        "text": seg["text"],
        "frame_paths": "|".join(frame_paths),
        "frame_indices": "|".join(map(str, frame_indices)),
        "label_status": "weak_label_auto",
    })

manifest_df = pd.DataFrame(manifest_rows)

print("Train-ready clips:", len(manifest_df))
display(manifest_df.head(20))

Train-ready clips: 732


,clip_id,video_id,segment_id,speaker_id,start_time,end_time,duration,start_frame,end_frame,num_frames,coverage,continuity,text,frame_paths,frame_indices,label_status
0,video001_seg000010_spk000,video001,10,0,50.76,52.78,2.02,1521,1582,62,1.0,1.0,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้,dataset\mouth_crops_final\video001\speaker_000...,1521|1522|1523|1524|1525|1526|1527|1528|1529|1...,weak_label_auto
1,video001_seg000011_spk000,video001,11,0,52.78,55.40,2.62,1581,1661,81,1.0,1.0,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน,dataset\mouth_crops_final\video001\speaker_000...,1581|1582|1583|1584|1585|1586|1587|1588|1589|1...,weak_label_auto
2,video001_seg000012_spk000,video001,12,0,55.40,56.92,1.52,1660,1706,47,1.0,1.0,อย่างเช่นการเอาไปอับเงินเดือน,dataset\mouth_crops_final\video001\speaker_000...,1660|1661|1662|1663|1664|1665|1666|1667|1668|1...,weak_label_auto
3,video001_seg000013_spk000,video001,13,0,56.92,58.76,1.84,1705,1762,58,1.0,1.0,หรือการเติบโตในหน้าที่การงาน,dataset\mouth_crops_final\video001\speaker_000...,1705|1706|1707|1708|1709|1710|1711|1712|1713|1...,weak_label_auto
4,video001_seg000016_spk001,video001,16,1,67.58,69.54,1.96,2025,2085,61,1.0,1.0,เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ,dataset\mouth_crops_final\video001\speaker_001...,2025|2026|2027|2028|2029|2030|2031|2032|2033|2...,weak_label_auto
5,video001_seg000017_spk001,video001,17,1,69.54,72.02,2.48,2084,2159,76,1.0,1.0,เหมือนเรามีครูส่วนตัวที่ไปทุกที่กับเรา,dataset\mouth_crops_final\video001\speaker_001...,2084|2085|2086|2087|2088|2089|2090|2091|2092|2...,weak_label_auto
6,video001_seg000018_spk001,video001,18,1,72.02,74.50,2.48,2158,2233,76,1.0,1.0,เรียนได้แบบ 24 ชั่วโมงเลย สะดวกมาก,dataset\mouth_crops_final\video001\speaker_001...,2158|2159|2160|2161|2162|2163|2164|2165|2166|2...,weak_label_auto
7,video001_seg000019_spk001,video001,19,1,74.50,75.48,0.98,2232,2263,32,1.0,1.0,ใช่ค่ะ,dataset\mouth_crops_final\video001\speaker_001...,2232|2233|2234|2235|2236|2237|2238|2239|2240|2...,weak_label_auto
8,video001_seg000020_spk001,video001,20,1,75.48,78.52,3.04,2262,2354,93,1.0,1.0,คือพี่ลองเล่นมาแล้ว แล้วพี่ชอบที่เขาจะมี AI,dataset\mouth_crops_final\video001\speaker_001...,2262|2263|2264|2265|2266|2267|2268|2269|2270|2...,weak_label_auto
9,video001_seg000021_spk001,video001,21,1,78.52,79.88,1.36,2353,2395,43,1.0,1.0,เหมือนก็ว่ามาประเมินก่อน,dataset\mouth_crops_final\video001\speaker_001...,2353|2354|2355|2356|2357|2358|2359|2360|2361|2...,weak_label_auto


## Cell 8 — Dataset Summary

In [8]:
if len(manifest_df) == 0:
    raise RuntimeError("ไม่มี clip ที่ผ่าน QA")

print("Clips by speaker:")
display(manifest_df["speaker_id"].value_counts())

print("\nDuration summary:")
display(manifest_df["duration"].describe())

print("\nFrame count summary:")
display(manifest_df["num_frames"].describe())

print("\nTotal training duration (minutes):",
      round(manifest_df["duration"].sum()/60, 2))

Clips by speaker:


speaker_id
1    437
0    295
Name: count, dtype: int64


Duration summary:


count    732.000000
mean       2.709467
std        1.141241
min        0.480000
25%        1.900000
50%        2.500000
75%        3.340000
max        7.760000
Name: duration, dtype: float64


Frame count summary:


count    732.000000
mean      83.132514
std       34.219660
min       16.000000
25%       59.000000
50%       77.000000
75%      102.000000
max      234.000000
Name: num_frames, dtype: float64


Total training duration (minutes): 33.06


## Cell 9 — Verify Paths

In [9]:
from pathlib import Path

missing = []
checked = 0

for _, row in manifest_df.iterrows():
    for rel in str(row["frame_paths"]).split("|"):
        checked += 1
        p = Path(rel)
        full = p if p.is_absolute() else PROJECT_ROOT / p

        if not full.exists():
            missing.append(str(full))
            if len(missing) >= 50:
                break

    if len(missing) >= 50:
        break

print("Checked paths:", checked)
print("Missing paths:", len(missing))

if missing:
    for p in missing[:10]:
        print(p)
else:
    print("All checked paths exist")

Checked paths: 60853
Missing paths: 0
All checked paths exist


## Cell 10 — Save

In [10]:
QA_CSV = OUTPUT_ROOT / "segment_qa.csv"
MANIFEST_CSV = OUTPUT_ROOT / "manifest_train_ready.csv"
ACCEPTED_CSV = OUTPUT_ROOT / "accepted_segments.csv"
REJECTED_CSV = OUTPUT_ROOT / "rejected_segments.csv"

qa_df.to_csv(QA_CSV, index=False, encoding="utf-8-sig")
manifest_df.to_csv(MANIFEST_CSV, index=False, encoding="utf-8-sig")
qa_df[qa_df["status"]=="ACCEPT"].to_csv(ACCEPTED_CSV, index=False, encoding="utf-8-sig")
qa_df[qa_df["status"]=="REJECT"].to_csv(REJECTED_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", QA_CSV)
print(" -", MANIFEST_CSV)
print(" -", ACCEPTED_CSV)
print(" -", REJECTED_CSV)

Saved:
 - D:\Face_detection\dataset\final_dataset\video001\segment_qa.csv
 - D:\Face_detection\dataset\final_dataset\video001\manifest_train_ready.csv
 - D:\Face_detection\dataset\final_dataset\video001\accepted_segments.csv
 - D:\Face_detection\dataset\final_dataset\video001\rejected_segments.csv


## Cell 11 — Preview Samples

In [11]:
sample_n = min(20, len(manifest_df))
sample_df = manifest_df.sample(
    n=sample_n,
    random_state=42
).sort_values("start_time")

display(sample_df[[
    "clip_id","speaker_id","duration","num_frames",
    "coverage","continuity","text"
]])

,clip_id,speaker_id,duration,num_frames,coverage,continuity,text
33,video001_seg000048_spk001,1,2.48,76,1.0,1.0,อยากฝึกการตอบโต๊ะแบบตัวๆ แต่ว่าบางที
39,video001_seg000059_spk000,0,1.92,60,1.0,1.0,หลังจากนั้นให้ใส่ Code TMI
66,video001_seg000097_spk000,0,5.66,171,1.0,1.0,แต่เรื่องมันไม่จบ เพราะแฟนหนุ่มกลับส่งค้อความญ...
72,video001_seg000103_spk000,0,3.18,97,1.0,1.0,เธอถึงกับนั่งร้องไห้ และพิมผลขอโทษให้เขาซำๆ
78,video001_seg000109_spk000,0,3.64,111,1.0,1.0,อยากไปหาศิลปินคนเดิมอีกครั้ง โดยไม่ได้คิดอะไร
164,video001_seg000322_spk000,0,2.32,72,1.0,1.0,ได้จองที่น сп์ข้างๆของเขาไว้ตัวเอง
181,video001_seg000340_spk000,0,1.78,55,1.0,1.0,ขอให้มีความสุขในวันเกริดนะ Matt
300,video001_seg000562_spk000,0,1.82,57,1.0,1.0,อะไร ? เขากลับไม่ยอมบอก narcissist
355,video001_seg000697_spk000,0,4.72,144,1.0,1.0,ถ้ามีใครพูดถึงรูปร่างแฟนของฉันในทางที่น่าสงสัย...
356,video001_seg000698_spk000,0,2.10,65,1.0,1.0,ฉันคงเอาเรื่องให้ถึงที่สุดแน่ๆ


# ขั้นต่อไป — 07 Dataset Split

หลัง Notebook นี้ผ่าน จะได้:
`dataset/final_dataset/video001/manifest_train_ready.csv`

จากนั้นทำ train/val/test split แล้วค่อยเริ่ม baseline training